# Câu 3 – Toán Tử LBP (Local Binary Patterns)
## Xử Lý Ảnh Số

---

**Đề bài:**

Cho 1 ảnh màu I kích thước n × m, chuyển đổi ảnh I thành ảnh xám. Dùng toán tử LBP (Local Binary Patterns) để biến đổi ảnh I theo các yêu cầu sau:
- Neighbors **P = 8**, R = 1 và R = 2
- Neighbors **P = 16**, R = 2 và R = 3
- Neighbors **P = 24**, R = 3

**Ghi chú:** Trường hợp P = 16 (hoặc P = 24) thì tách chuỗi nhị phân thành 2 phần (hoặc 3 phần). Tức là mỗi phần chuỗi có 8 bits. Sau đó, chỉ lấy phần có **giá trị lớn nhất** gán cho pixel đang xét (center pixel).

---

| STT | P | R | Số nhóm | Kết quả |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 8 | 1 | 1 | $V_1$ |
| 2 | 8 | 2 | 1 | $V_1$ |
| 3 | 16 | 2 | 2 | $\max(V_1, V_2)$ |
| 4 | 16 | 3 | 2 | $\max(V_1, V_2)$ |
| 5 | 24 | 3 | 3 | $\max(V_1, V_2, V_3)$ |

---
## Bước 1 – Import thư viện

In [1]:
import os                          # Thao tác đường dẫn file, thư mục
import time                        # Đo thời gian chạy từng cấu hình
import numpy as np                 # Tính toán ma trận số học (mảng đa chiều)
import matplotlib.pyplot as plt    # Vẽ ảnh và biểu đồ histogram
from PIL import Image              # Đọc ảnh màu, chuyển sang ảnh xám
from scipy.ndimage import map_coordinates  # Nội suy song tuyến toàn ảnh (vectorized)


---
## Bước 2 – Đọc ảnh và chuyển sang ảnh xám

Đề bài yêu cầu từ ảnh màu I, chuyển sang ảnh xám trước rồi mới áp dụng LBP.

**Công thức chuyển đổi chuẩn (ITU-R BT.601):**
$$\text{Gray} = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

Thư viện PIL tự động áp dụng công thức này khi gọi `.convert('L')`.

**Lưu ý về pixel biên (border):** Khi áp dụng LBP với bán kính R, các pixel cách mép ảnh dưới R pixel **không có đủ lân cận** để tính → gán giá trị 0.

$$\text{Số hàng/cột biên bỏ qua} = \lceil R \rceil$$

---
## Bước 3 – Các hàm tính LBP

### 3.1. Hàm nội suy song tuyến (Bilinear Interpolation)

Tọa độ điểm lân cận thứ $p$ trên vòng tròn bán kính $R$:
$$x_p = cx + R \cdot \cos\!\left(\frac{2\pi p}{P}\right) \qquad y_p = cy - R \cdot \sin\!\left(\frac{2\pi p}{P}\right)$$

*(Dấu trừ ở $y_p$ vì trục Y ảnh đi từ trên xuống dưới)*

Khi tọa độ là số thực (không phải nguyên), ta nội suy từ 4 pixel góc gần nhất:

$$g_p = (1-dy)(1-dx)\cdot f_{00} + (1-dy)\cdot dx\cdot f_{01} + dy\cdot(1-dx)\cdot f_{10} + dy\cdot dx\cdot f_{11}$$

trong đó $dx = x_p - \lfloor x_p \rfloor$, $\; dy = y_p - \lfloor y_p \rfloor$

### 3.2. Tạo chuỗi bit và tính giá trị thập phân

$$\text{bit}_p = \begin{cases} 1 & \text{nếu } g_p \geq g_c \\ 0 & \text{nếu } g_p < g_c \end{cases}$$

Quy đổi từng nhóm 8-bit sang thập phân (bit_0 = LSB):
$$V = \sum_{i=0}^{7} \text{bit}_i \times 2^i \in [0,\; 255]$$

### 3.3. Xử lý đặc biệt theo đề bài:
- **P = 8** → 1 nhóm → $\text{LBP} = V_1$
- **P = 16** → 2 nhóm → $\text{LBP} = \max(V_1, V_2)$
- **P = 24** → 3 nhóm → $\text{LBP} = \max(V_1, V_2, V_3)$

In [2]:
from scipy.ndimage import map_coordinates  # Import hàm nội suy scipy (dùng trong phiên bản tối ưu)

# ─────────────────────────────────────────────────────────────
# HÀM 1: Nội suy song tuyến tại 1 tọa độ thực
# ─────────────────────────────────────────────────────────────
def noi_suy_song_tuyen(anh, y, x):
    """
    Tính giá trị mức sáng tại tọa độ thực (x, y) bằng nội suy song tuyến.

    Vì tọa độ điểm lân cận trên vòng tròn là số thực (vd: x=1.7, y=0.3),
    ta không thể lấy pixel trực tiếp → phải ước lượng từ 4 pixel nguyên
    gần nhất xung quanh.

    Công thức nội suy song tuyến:
        gp = (1-dy)(1-dx)*f00 + (1-dy)*dx*f01
           +    dy*(1-dx)*f10 +    dy *dx*f11
    Trong đó:
        f00 = pixel góc trái-trên   (x0, y0)
        f01 = pixel góc phải-trên   (x1, y0)
        f10 = pixel góc trái-dưới   (x0, y1)
        f11 = pixel góc phải-dưới   (x1, y1)
        dx  = phần thập phân theo trục X (khoảng cách từ x0 đến x)
        dy  = phần thập phân theo trục Y (khoảng cách từ y0 đến y)
    """
    rows, cols = anh.shape   # Lấy số hàng, số cột của ảnh để kiểm tra biên

    # Tính tọa độ nguyên bao quanh tọa độ thực (x, y)
    x0 = int(np.floor(x))   # Tọa độ cột phía trái  (làm tròn xuống)
    x1 = x0 + 1             # Tọa độ cột phía phải  (x0 + 1)
    y0 = int(np.floor(y))   # Tọa độ hàng phía trên (làm tròn xuống)
    y1 = y0 + 1             # Tọa độ hàng phía dưới (y0 + 1)

    # Giữ các tọa độ trong biên ảnh (tránh truy cập ngoài mảng)
    # Nếu x0 < 0 thì dùng 0; nếu x0 >= cols thì dùng cols-1
    x0 = np.clip(x0, 0, cols-1)
    x1 = np.clip(x1, 0, cols-1)
    y0 = np.clip(y0, 0, rows-1)
    y1 = np.clip(y1, 0, rows-1)

    # Tính phần thập phân (khoảng cách từ x0, y0 đến tọa độ thực)
    dx = x - np.floor(x)   # Vd: x=1.7 → dx=0.7 (gần x1 hơn)
    dy = y - np.floor(y)   # Vd: y=0.3 → dy=0.3 (gần y0 hơn)

    # Lấy mức sáng của 4 pixel góc gần nhất
    f00 = anh[y0, x0]   # Góc trái-trên
    f01 = anh[y0, x1]   # Góc phải-trên
    f10 = anh[y1, x0]   # Góc trái-dưới
    f11 = anh[y1, x1]   # Góc phải-dưới

    # Áp dụng công thức nội suy song tuyến → trả về giá trị ước lượng
    return (1-dy)*(1-dx)*f00 + (1-dy)*dx*f01 \
         +    dy *(1-dx)*f10 +    dy *dx*f11


# ─────────────────────────────────────────────────────────────
# HÀM 2: Tính LBP cho 1 pixel trung tâm
# ─────────────────────────────────────────────────────────────
def tinh_lbp_mot_pixel(anh_xam, cy, cx, P, R):
    """
    Tính giá trị LBP cho pixel tại vị trí (cy, cx) trên ảnh xám.

    Tham số:
        anh_xam : ma trận ảnh xám (float64)
        cy, cx  : tọa độ hàng, cột của pixel trung tâm
        P       : số điểm lân cận xét trên vòng tròn
        R       : bán kính vòng tròn lân cận (đơn vị pixel)

    Trả về: giá trị LBP trong [0, 255]
    """
    gc = anh_xam[cy, cx]   # Lấy mức sáng của pixel TRUNG TÂM đang xét
    bits = []               # Danh sách lưu chuỗi bit nhị phân (0 hoặc 1)

    # Duyệt qua từng điểm lân cận thứ p (p = 0, 1, ..., P-1)
    for p in range(P):
        # Bước 2a: Tính góc của điểm lân cận thứ p trên vòng tròn
        # Chia đều 360° thành P phần → mỗi phần cách nhau 2π/P radian
        goc = 2 * np.pi * p / P

        # Tính tọa độ (xp, yp) của điểm lân cận thứ p
        # xp = cx + R*cos(goc): dịch sang phải/trái theo trục X
        # yp = cy - R*sin(goc): dấu TRỪ vì trục Y ảnh đi từ trên→xuống
        #                        (ngược chiều toán học từ dưới→lên)
        xp = cx + R * np.cos(goc)
        yp = cy - R * np.sin(goc)

        # Bước 2b: Lấy mức sáng tại (xp, yp) bằng nội suy song tuyến
        # vì (xp, yp) là số thực, không phải tọa độ nguyên
        gp = noi_suy_song_tuyen(anh_xam, yp, xp)

        # Bước 2c: So sánh mức sáng lân cận với tâm → tạo 1 bit
        # Nếu lân cận sáng hơn hoặc bằng tâm → bit = 1, ngược lại → bit = 0
        bits.append(1 if gp >= gc else 0)

    # Bước 3: Chia chuỗi P bit thành các nhóm 8-bit rồi đổi sang thập phân
    # Vd: P=16 → 2 nhóm; P=24 → 3 nhóm; P=8 → 1 nhóm
    so_nhom = P // 8       # Số nhóm 8-bit
    gia_tri_nhom = []      # Danh sách lưu giá trị thập phân của từng nhóm

    for g in range(so_nhom):
        nhom = bits[g*8 : (g+1)*8]   # Cắt ra 8 bit của nhóm thứ g
        # Quy đổi nhị phân → thập phân: bit[0]*2^0 + bit[1]*2^1 + ...
        # bit_0 là LSB (Least Significant Bit = bit có giá trị nhỏ nhất = 2^0)
        val = sum(b * (2**i) for i, b in enumerate(nhom))
        gia_tri_nhom.append(val)

    # Bước 4: Lấy giá trị LỚN NHẤT trong các nhóm làm giá trị LBP cuối cùng
    # Mục đích: đảm bảo kết quả luôn nằm trong [0, 255] để lưu ảnh 8-bit
    return max(gia_tri_nhom)


# ─────────────────────────────────────────────────────────────
# HÀM 3a: Tính LBP toàn ảnh – Phiên bản gốc (for loop)
# ─────────────────────────────────────────────────────────────
def tinh_lbp_toan_anh(anh_xam, P, R):
    """
    [PHIÊN BẢN GỐC - Vòng lặp for]
    Áp dụng LBP lên toàn bộ ảnh xám bằng vòng lặp tuần tự.
    Pixel trong vùng biên (cách mép < R pixel) được gán = 0.
    Lưu ý: Chậm trên ảnh lớn vì duyệt từng pixel một.
    """
    rows, cols = anh_xam.shape                  # Lấy kích thước ảnh (H×W)
    anh_lbp    = np.zeros((rows, cols), dtype=np.uint8)  # Tạo ảnh kết quả rỗng (toàn 0)
    bien       = int(np.ceil(R))                # Số pixel biên bỏ qua mỗi phía = ceil(R)
                                                # Vd: R=1→bien=1; R=2.5→bien=3

    # Duyệt từng pixel trong vùng nội (bỏ qua vùng biên)
    for cy in range(bien, rows - bien):         # Duyệt từng hàng (bỏ bien hàng đầu/cuối)
        for cx in range(bien, cols - bien):     # Duyệt từng cột (bỏ bien cột đầu/cuối)
            # Tính LBP cho pixel (cy, cx) và lưu vào ảnh kết quả
            anh_lbp[cy, cx] = tinh_lbp_mot_pixel(anh_xam, cy, cx, P, R)

    return anh_lbp   # Trả về ảnh LBP (cùng kích thước ảnh đầu vào)


# ─────────────────────────────────────────────────────────────
# HÀM 3b: Tính LBP toàn ảnh – Phiên bản tối ưu (scipy)
# ─────────────────────────────────────────────────────────────
def tinh_lbp_toan_anh_scipy(anh_xam, P, R):
    """
    [PHIÊN BẢN TỐI ƯU - Vector hóa NumPy + scipy]
    Áp dụng LBP lên toàn bộ ảnh xám, cùng kết quả với tinh_lbp_toan_anh().

    Bản chất thuật toán KHÔNG thay đổi:
      - Cùng công thức nội suy song tuyến (order=1 trong map_coordinates)
      - Cùng cách so sánh lân cận với tâm
      - Cùng cách gom nhóm 8-bit và lấy MAX
    Khác biệt duy nhất: xử lý toàn bộ ảnh cùng lúc thay vì từng pixel một.
    Tốc độ nhanh hơn ~100 lần trên ảnh lớn.
    """
    rows, cols = anh_xam.shape   # Lấy kích thước ảnh

    # Tạo lưới tọa độ (y, x) cho TẤT CẢ pixel trong ảnh cùng lúc
    # grid_y[i,j] = i  (chỉ số hàng của pixel tại vị trí (i,j))
    # grid_x[i,j] = j  (chỉ số cột của pixel tại vị trí (i,j))
    # Vd: ảnh 3×3 → grid_y=[[0,0,0],[1,1,1],[2,2,2]], grid_x=[[0,1,2],[0,1,2],[0,1,2]]
    grid_y, grid_x = np.mgrid[0:rows, 0:cols]   # shape: (rows, cols)

    bits_all = []   # Danh sách lưu P ma trận bit (mỗi ma trận tương ứng 1 điểm lân cận)

    # Duyệt qua từng điểm lân cận thứ p (chỉ lặp P lần, không phải rows×cols×P)
    for p in range(P):
        # Tính góc và tọa độ lân cận (giống hệt tinh_lbp_mot_pixel nhưng cho toàn ảnh)
        goc = 2 * np.pi * p / P                   # Góc của điểm lân cận thứ p

        # ny, nx là tọa độ của điểm lân cận thứ p cho TẤT CẢ pixel cùng lúc
        # ny[i,j] = tọa độ y của lân cận p của pixel (i,j)
        # nx[i,j] = tọa độ x của lân cận p của pixel (i,j)
        ny  = grid_y - R * np.sin(goc)
        nx  = grid_x + R * np.cos(goc)

        # Nội suy song tuyến cho TẤT CẢ điểm lân cận cùng lúc (1 lệnh duy nhất)
        # map_coordinates: tương đương gọi noi_suy_song_tuyen() cho mọi pixel
        # order=1       → bilinear interpolation (nội suy song tuyến)
        # mode='nearest'→ pixel biên lấy giá trị pixel gần nhất (thay vì = 0)
        gp = map_coordinates(anh_xam, [ny, nx], order=1, mode='nearest')
        # gp là ma trận (rows×cols): gp[i,j] = mức sáng lân cận p của pixel (i,j)

        # So sánh toàn bộ ma trận lân cận với ma trận pixel trung tâm (1 lệnh)
        # Kết quả: ma trận bit True/False → ép sang 0/1 (uint8)
        # bits_all[p][i,j] = 1 nếu lân cận p của pixel (i,j) sáng hơn hoặc bằng tâm
        bits_all.append((gp >= anh_xam).astype(np.uint8))

    # ── Gom nhóm 8-bit và lấy MAX ──────────────────────────────────
    so_nhom = P // 8   # Số nhóm: P=8→1, P=16→2, P=24→3

    # Mảng lũy thừa của 2: [1, 2, 4, 8, 16, 32, 64, 128]
    # Dùng để quy đổi nhị phân → thập phân nhanh hơn
    pow2 = np.array([1, 2, 4, 8, 16, 32, 64, 128], dtype=np.uint16)

    # groups[g] = ma trận (rows×cols) lưu giá trị thập phân của nhóm g, cho toàn ảnh
    groups = np.zeros((so_nhom, rows, cols), dtype=np.uint16)

    for g in range(so_nhom):       # Duyệt từng nhóm 8-bit
        for i in range(8):         # Duyệt từng bit trong nhóm
            # Cộng dồn: bit thứ i của nhóm g × 2^i, cho toàn ảnh cùng lúc
            # bits_all[g*8+i] là ma trận bit thứ (g*8+i) trong toàn bộ P bit
            groups[g] += bits_all[g*8 + i] * pow2[i]

    # Lấy giá trị MAX giữa các nhóm theo từng pixel (axis=0 = so sánh giữa các nhóm)
    # groups.max(axis=0)[i,j] = max(groups[0][i,j], groups[1][i,j], ...)
    # Ép sang uint8 để giá trị nằm trong [0, 255]
    return groups.max(axis=0).astype(np.uint8)


print('Đã định nghĩa xong các hàm LBP!')
print('  - tinh_lbp_toan_anh()       : phiên bản gốc (for loop)')
print('  - tinh_lbp_toan_anh_scipy() : phiên bản tối ưu (vector hóa)')


Đã định nghĩa xong các hàm LBP!
  - tinh_lbp_toan_anh()       : phiên bản gốc (for loop)
  - tinh_lbp_toan_anh_scipy() : phiên bản tối ưu (vector hóa)


---
## Bước 4 – Cấu hình và quét danh sách ảnh

In [3]:
# Đường dẫn thư mục chứa ảnh đầu vào
INPUT_DIR  = os.path.join('anh_xlas', 'anh_xlas')   # Thư mục: anh_xlas/anh_xlas/

# Thư mục lưu ảnh kết quả LBP và histogram
OUTPUT_DIR = 'ket_qua_lbp'
os.makedirs(OUTPUT_DIR, exist_ok=True)   # Tạo thư mục nếu chưa tồn tại (exist_ok=True: không báo lỗi nếu đã có)

# Danh sách 5 cấu hình LBP theo yêu cầu đề bài
# Mỗi phần tử là tuple (P, R, nhãn_hiển_thị)
# P = số điểm lân cận | R = bán kính vòng tròn | nhãn = chuỗi dùng để in/đặt tiêu đề
CAU_HINH = [
    (8,  1, 'P=8,  R=1'),   # 8 điểm lân cận, bán kính 1 pixel (liền kề)
    (8,  2, 'P=8,  R=2'),   # 8 điểm lân cận, bán kính 2 pixel
    (16, 2, 'P=16, R=2'),   # 16 điểm lân cận, bán kính 2 pixel
    (16, 3, 'P=16, R=3'),   # 16 điểm lân cận, bán kính 3 pixel
    (24, 3, 'P=24, R=3'),   # 24 điểm lân cận, bán kính 3 pixel
]

# Các đuôi file ảnh được hỗ trợ
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

# Quét tất cả file trong INPUT_DIR, lọc theo đuôi file, sắp xếp theo tên
ds_anh = sorted([
    f for f in os.listdir(INPUT_DIR)       # Liệt kê tất cả file trong thư mục
    if f.lower().endswith(EXTENSIONS)      # Chỉ giữ file có đuôi là ảnh
])

# In danh sách ảnh tìm được
print(f'Tìm thấy {len(ds_anh)} ảnh cần xử lý:')
for ten in ds_anh:
    print(f'  - {ten}')


Tìm thấy 10 ảnh cần xử lý:
  - image_01.jpg
  - image_02.jpg
  - image_03.jpg
  - image_04.jpg
  - image_05.jpg
  - image_06.jpg
  - image_07.jpg
  - image_08.jpg
  - image_09.jpg
  - image_10.jpg


---
## Bước 5 – Xử lý LBP hàng loạt cho 10 ảnh

Với mỗi ảnh, chương trình thực hiện tuần tự:
1. Đọc ảnh → chuyển sang ảnh xám (PIL tự dùng công thức Gray = 0.299R + 0.587G + 0.114B)
2. Áp dụng LBP với cả 5 cấu hình bằng vòng lặp
3. Lưu ảnh kết quả và biểu đồ histogram

In [4]:
t_bat_dau = time.time()   # Ghi lại thời điểm bắt đầu toàn bộ quá trình xử lý

# ── VÒNG LẶP CHÍNH: Duyệt lần lượt từng ảnh trong danh sách ───────────
# enumerate(ds_anh, 1): lấy cả chỉ số (bắt đầu từ 1) và tên file
for idx, ten_anh in enumerate(ds_anh, 1):
    duong_dan = os.path.join(INPUT_DIR, ten_anh)   # Ghép đường dẫn đầy đủ đến file ảnh
    print(f'\n[{idx}/{len(ds_anh)}] Đang xử lý: {ten_anh}')   # In tiến trình

    try:   # Bọc try/except để nếu 1 ảnh lỗi thì vẫn tiếp tục xử lý ảnh tiếp theo

        # ── BƯỚC 1: Đọc ảnh và chuyển sang ảnh xám ────────────────────────
        img_pil = Image.open(duong_dan).convert('L')
        # Image.open(): mở file ảnh màu RGB từ đĩa
        # .convert('L'): chuyển sang ảnh xám (L = Luminance)
        #   Công thức: Gray = 0.299*R + 0.587*G + 0.114*B  (chuẩn ITU-R BT.601)
        #   Kết quả: mỗi pixel là 1 số nguyên [0, 255]

        img_xam = np.array(img_pil, dtype=np.float64)
        # Chuyển ảnh PIL thành mảng NumPy 2D kiểu float64
        # Dùng float64 (thay vì uint8) để tránh mất độ chính xác khi nội suy song tuyến

        H, W = img_xam.shape   # Lấy chiều cao H (số hàng) và chiều rộng W (số cột)
        print(f'    Kích thước: {H} x {W} pixels')

        ten_goc = os.path.splitext(ten_anh)[0]
        # os.path.splitext: tách tên file và đuôi file
        # Vd: "image_01.jpg" → ("image_01", ".jpg") → lấy [0] = "image_01"
        # Dùng để đặt tên file kết quả lưu ra

        # ── BƯỚC 2: Áp dụng LBP với cả 5 cấu hình ─────────────────────────
        ket_qua = {}    # Dictionary lưu kết quả: {nhãn: ảnh_lbp}
        t0_anh  = time.time()   # Ghi thời điểm bắt đầu xử lý ảnh này

        # Duyệt lần lượt 5 cấu hình (P, R)
        for P, R, nhan in CAU_HINH:
            t0 = time.time()   # Bắt đầu đo thời gian cho cấu hình này

            # GỌI HÀM TÍNH LBP CHÍNH (phiên bản tối ưu scipy)
            anh_lbp = tinh_lbp_toan_anh_scipy(img_xam, P, R)
            # anh_lbp: ma trận uint8 (H×W), mỗi giá trị là đặc trưng LBP [0, 255]

            ket_qua[nhan] = anh_lbp   # Lưu kết quả vào dict với key là nhãn

            # In thống kê: thời gian chạy + min/max/mean của ảnh LBP
            print(f'    {nhan}: {time.time()-t0:.2f}s  '
                  f'| min={anh_lbp.min():3d}  max={anh_lbp.max():3d}  mean={anh_lbp.mean():.1f}')

        print(f'    => Tổng thời gian: {time.time()-t0_anh:.2f}s')

        # ── BƯỚC 3: Vẽ và lưu ảnh so sánh (lưới 2×3: ảnh gốc xám + 5 LBP) ─
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        # Tạo figure có lưới 2 hàng × 3 cột = 6 ô vẽ
        # figsize=(16, 10): kích thước figure tính bằng inch

        fig.suptitle(f'Kết quả LBP – {ten_anh}', fontsize=15, fontweight='bold')
        # Tiêu đề chung cho toàn bộ figure

        ax_list = axes.ravel()
        # axes là mảng 2D shape (2,3), .ravel() làm phẳng thành 1D [ax0..ax5]
        # Để truy cập từng ô theo chỉ số: ax_list[0], ax_list[1], ...

        # Ô đầu tiên (ax_list[0]): hiển thị ảnh xám gốc
        ax_list[0].imshow(img_xam, cmap='gray', vmin=0, vmax=255)
        # cmap='gray': dùng bảng màu xám | vmin/vmax: chuẩn hóa hiển thị [0,255]
        ax_list[0].set_title('Ảnh xám gốc', fontsize=12, fontweight='bold')
        ax_list[0].axis('off')   # Tắt hiển thị trục tọa độ

        # 5 ô còn lại (ax_list[1..5]): hiển thị 5 ảnh LBP theo từng cấu hình
        for i, (nhan, anh_lbp) in enumerate(ket_qua.items()):
            ax_list[i+1].imshow(anh_lbp, cmap='gray', vmin=0, vmax=255)
            ax_list[i+1].set_title(nhan, fontsize=12, fontweight='bold')
            ax_list[i+1].axis('off')

        plt.tight_layout()   # Tự động điều chỉnh khoảng cách giữa các ô

        # Lưu ảnh so sánh ra file PNG
        path_anh = os.path.join(OUTPUT_DIR, f'{ten_goc}_lbp.png')
        plt.savefig(path_anh, dpi=120, bbox_inches='tight')
        # dpi=120: độ phân giải 120 chấm/inch (ảnh rõ hơn default 100)
        # bbox_inches='tight': cắt bỏ khoảng trắng thừa xung quanh
        plt.close()   # Đóng figure để giải phóng bộ nhớ

        # ── BƯỚC 4: Vẽ và lưu histogram phân bố giá trị LBP ───────────────
        fig, axes = plt.subplots(1, 5, figsize=(22, 4))
        # Tạo figure có 1 hàng × 5 cột = 5 ô histogram (1 ô/cấu hình)

        fig.suptitle(f'Histogram LBP – {ten_anh}', fontsize=13, fontweight='bold')

        # Vẽ histogram cho từng cấu hình
        for ax, (nhan, anh_lbp) in zip(axes, ket_qua.items()):
            px = anh_lbp[anh_lbp > 0].ravel()
            # anh_lbp > 0: tạo mask boolean, loại bỏ pixel biên (giá trị = 0)
            # .ravel(): làm phẳng ma trận 2D thành mảng 1D để vẽ histogram

            ax.hist(px, bins=32, range=(0, 255), color='steelblue',
                    edgecolor='white', linewidth=0.5)
            # bins=32   : chia [0,255] thành 32 cột histogram
            # range     : giới hạn trục X
            # color     : màu cột histogram
            # edgecolor : màu viền cột

            ax.set_title(nhan, fontsize=10, fontweight='bold')
            ax.set_xlabel('Giá trị LBP')    # Nhãn trục X
            ax.set_ylabel('Số pixel')         # Nhãn trục Y
            ax.set_facecolor('#F5F5F5')       # Màu nền ô vẽ (xám nhạt)

        plt.tight_layout()

        # Lưu histogram ra file PNG
        path_hist = os.path.join(OUTPUT_DIR, f'{ten_goc}_histogram.png')
        plt.savefig(path_hist, dpi=120, bbox_inches='tight')
        plt.close()   # Đóng figure giải phóng bộ nhớ

        print(f'    Đã lưu: {path_anh}')
        print(f'    Đã lưu: {path_hist}')

    except Exception as e:
        # Nếu có lỗi bất kỳ (file hỏng, không đọc được...) thì in thông báo
        # và tiếp tục xử lý ảnh tiếp theo (không dừng toàn bộ chương trình)
        print(f'    LỖI khi xử lý {ten_anh}: {e}')

# In tổng kết sau khi xử lý xong tất cả ảnh
print(f'\n Hoàn thành {len(ds_anh)} ảnh sau {time.time()-t_bat_dau:.2f} giây!')
print(f'Kết quả lưu tại thư mục: "{OUTPUT_DIR}/"')



[1/10] Đang xử lý: image_01.jpg
    Kích thước: 1080 x 1920 pixels
    P=8,  R=1: 1.33s  | min=  0  max=255  mean=152.5
    P=8,  R=2: 0.94s  | min=  0  max=255  mean=150.1
    P=16, R=2: 1.77s  | min=  0  max=255  mean=190.0
    P=16, R=3: 1.79s  | min=  0  max=255  mean=187.9
    P=24, R=3: 2.86s  | min=  0  max=255  mean=206.5
    => Tổng thời gian: 8.70s
    Đã lưu: ket_qua_lbp/image_01_lbp.png
    Đã lưu: ket_qua_lbp/image_01_histogram.png

[2/10] Đang xử lý: image_02.jpg
    Kích thước: 1080 x 1920 pixels
    P=8,  R=1: 1.24s  | min=  0  max=255  mean=132.7
    P=8,  R=2: 1.27s  | min=  0  max=255  mean=129.2
    P=16, R=2: 2.82s  | min=  0  max=255  mean=177.9
    P=16, R=3: 2.53s  | min=  0  max=255  mean=176.6
    P=24, R=3: 3.22s  | min=  0  max=255  mean=197.0
    => Tổng thời gian: 11.09s
    Đã lưu: ket_qua_lbp/image_02_lbp.png
    Đã lưu: ket_qua_lbp/image_02_histogram.png

[3/10] Đang xử lý: image_03.jpg
    Kích thước: 1080 x 1920 pixels
    P=8,  R=1: 1.86s  | min=  0